## Imports

In [1]:
import numpy as np
import os

cwd = os.getcwd()
parent = os.path.dirname(cwd)
grandparent = os.path.dirname(parent)

data_dir =  f"{parent}/data/"


## Make helper functions

In [2]:
def encode( sequence, symbols):
    
    enc = [0] * len(sequence)
    
    for i in range(len(sequence)):
        enc[i] = symbols.find(sequence[i])
    
    return(enc)



## Prepare data

In [16]:
#load protein sequences 
seqs = []
location = []
encoding = {}

alphabet_file = data_dir + "alphabet"
alphabet = np.loadtxt(alphabet_file, dtype=str)

for i, AA in enumerate(alphabet):
    encoding[AA] = i

with open(data_dir + "signalpeptide.csv") as f:
    #print(f.read().split())
    for i,line in enumerate(f.read().split()):
        #dont include the first line
        if i > 1:
            current_sequence = line.split(",")[0]
            enc_sequence = []
            for AA in current_sequence:
                #print(encoding[AA])
                enc_sequence.append( encoding[AA] )
            #encode the sequence to be between 1 and 20 (corresponding to an AA)
            seqs.append(enc_sequence)
            location.append(line.split(",")[1])

#output:
## two lists, 
### one with 30-mer peptides
### one with translocation
print(seqs[:3])

[[12, 15, 11, 14, 14, 3, 10, 10, 10, 1, 10, 10, 1, 7, 0, 14, 1, 5, 1, 19, 4, 16, 10, 13, 9, 9, 7, 13, 11, 13, 16, 13, 13, 19, 15, 9, 12, 9, 18, 17, 8, 19, 19, 7, 6, 14, 11, 6, 11, 7], [12, 0, 0, 15, 19, 0, 0, 0, 0, 1, 1, 10, 1, 1, 0, 9, 1, 1, 15, 14, 0, 17, 1, 7, 10, 15, 8, 1, 14, 10, 15, 15, 6, 14, 14, 0, 0, 11, 0, 15, 0, 19, 1, 0, 0, 13, 10, 2, 13, 13], [12, 10, 5, 19, 8, 1, 16, 7, 10, 7, 1, 10, 7, 19, 15, 10, 15, 11, 7, 10, 8, 8, 11, 0, 19, 10, 0, 19, 1, 1, 6, 3, 19, 2, 0, 17, 6, 1, 1, 0, 14, 10, 0, 14, 11, 8, 9, 11, 7, 9]]


## Initialize model architecture

Make the initial_propablity for each starting state (always P1), initialte emission propabilities, and initiate transition propabilities.

In [ ]:
sequence_length = len(seqs[0])

#sequence always starts with M, so initial propability for M is 1, and everything else is 0
#initial_prop = [0 if p != "M" else 1 for p in alphabet ]
#print(initial_prop)

#####states
states = [f"P{1}", "I1"]

for i in range(1, sequence_length-2):
    states.append(f"P{i+1}")
    states.append(f"I{i+1}")
    states.append(f"D{i+1}")

#make last 2 states, which are gonna be unique
states.append(f"P{sequence_length-1}")
states.append(f"I{sequence_length-1}")
states.append(f"P{sequence_length}")

#all states can go to end state
states.append("end")

### Emissions (each row is a state and each column is an emission from the alphabet)
#initial emission propabilty (equal propabilty for all AA)
p_initial_emission = [1/len(alphabet)]*len(alphabet) + [0]
d_emission = [0] *len(alphabet) + [1] # always ""
i_initial_emission = [1/len(alphabet)]*len(alphabet) + [0]

emission = np.array([
    p_initial_emission if state.startswith("P")
    else d_emission if state.startswith("D")
    else i_initial_emission if state.startswith("I")
    else d_emission  # end state
    for state in states
])


###initial transmission propability
transmission = np.zeros ( (len(states), len(states)) )

for i, s in enumerate(states):
    #if it is not the last state
    if i != len(states) -1:
        if s[0] == "P":
            #check if there exists an I state
            if states[i+1].startswith("I"):
                transmission[i,i+1] = 1
                #is it followed by a D state
                if states[i+2].startswith("D"):
                    transmission[i,i+2] = 1
                    #is it followed by a P state
                    if states[i+3].startswith("P"):
                        transmission[i,i+3] = 1
                elif states[i+2].startswith("P"):
                        transmission[i,i+2] = 1
        elif s[0] == "I":
            transmission[i,i] = 1
            #check if there exists an P state next
            if states[i+1].startswith("P"):
                transmission[i,i+1] = 1
            elif states[i+2].startswith("P"):
                transmission[i,i+1] = 1
        elif s[0] == "D":

            if states[i+2].startswith("P"):
                transmission[i,i+2] = 1
                if i < len(states) - 2:
                    if states[i+3].startswith("D"):
                        transmission[i,i+3] = 1
    else:
        transmission[i,i] = 1



[[0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 1. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]]


## Define model functions